In [ ]:
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, train_test_split, RandomizedSearchCV
import pickle as pkl
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure
from sklearn import preprocessing
pd.set_option('display.max_columns', None) 

Import the cleansed dataset for reporting

In [ ]:
#Define paths
notebook_dir = Path().resolve()

output_dir = notebook_dir / "outputs"
output_dir.mkdir(exist_ok=True)

csv_path = {output_dir}/ "atx_crash_data_2018-2026_clean.csv"

df_raw = pd.read_csv(csv_path)
print(f"Loaded CSV from: {csv_path}")

C:\Users\jacqueline.pielli\AppData\Local\Temp\ipykernel_41528\877512960.py:1: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv(r"C:\Users\jacqueline.pielli\Documents\Project Nova\BI to AI\atx_crash_data_2018-2026_cleansed.csv")


In [4]:
df_raw.head()

,ID,Crash ID,crash_fatal_fl,case_id,rpt_block_num,rpt_street_name,rpt_street_sfx,crash_speed_limit,road_constr_zone_fl,latitude,...,micromobility device_involved,e-scooter_involved,other_involved,passenger car_involved,bicycle_involved,large passenger vehicle_involved,train_involved,motor vehicle_involved,motorcycle_involved,pedestrian_involved
0,246606,16209332.0,False,180010160,12300,HARRIS BRANCH,PKWY,45.0,False,30.368405,...,False,False,False,True,False,False,False,False,False,False
1,245426,16182659.0,False,180010462,8300,BEN WHITE,BLVD,55.0,False,30.222877,...,False,False,False,True,False,False,False,False,False,False
2,245513,16185981.0,False,180010550,5800,LARK CREEK DR,DR,30.0,False,30.192214,...,False,False,False,True,False,False,False,False,False,False
3,245368,16180819.0,False,180010555,1700,E PARMER LN,LN,60.0,False,30.386948,...,False,False,False,True,False,False,False,False,False,False
4,278237,17979600.0,False,180010873,700,PARMER,LN,60.0,False,30.405337,...,False,False,False,True,False,False,False,False,False,False


Create the features dataset for prediction

In [ ]:
target = 'Estimated Total Comprehensive Cost'
drop_cols = ['ID','Crash ID','case_id','rpt_street_sfx','point','Address','crash_sev_id','crash_fatal_fl','death_cnt','Crash timestamp (US/Central)', 'rpt_block_num','rpt_street_name', target]
df_x = df_raw.drop(columns=drop_cols)
df_y = np.log1p(df_raw[target].astype(float))

In [6]:
df_x.head()

,crash_speed_limit,road_constr_zone_fl,latitude,longitude,onsys_fl,private_dr_fl,Location group,day_of_week,week_of_year,hour_of_day,micromobility device_involved,e-scooter_involved,other_involved,passenger car_involved,bicycle_involved,large passenger vehicle_involved,train_involved,motor vehicle_involved,motorcycle_involved,pedestrian_involved
0,45.0,False,30.368405,-97.613322,True,False,1.0,0,1,0,False,False,False,True,False,False,False,False,False,False
1,55.0,False,30.222877,-97.679100,True,False,2.0,0,1,2,False,False,False,True,False,False,False,False,False,False
2,30.0,False,30.192214,-97.731125,False,False,1.0,0,1,2,False,False,False,True,False,False,False,False,False,False
3,60.0,False,30.386948,-97.648715,True,False,1.0,0,1,2,False,False,False,True,False,False,False,False,False,False
4,60.0,False,30.405337,-97.664316,True,False,1.0,0,1,7,False,False,False,True,False,False,False,False,False,False


Skip next 5 cells if already trained model.

In [ ]:
#Create train and test data sets
x_train,x_test,y_train,y_test = train_test_split(df_x,df_y,test_size=.2,random_state=42)

In [ ]:
#Set parameters to be considered in the grid search
params = {
    'n_estimators':[25,50,100,200],
    'criterion': ['squared_error'],
    'max_depth': [3,5,10,None],
    'min_samples_leaf':[1,2,4],
    'max_features': ['sqrt','log2',None]
}

clf = GridSearchCV(RandomForestRegressor(),params,n_jobs=-1,cv=5)

In [ ]:
#Fit the model using the best parameters and save down the results of the grid search
clf.fit(x_train,y_train)

with open(notebook_dir /'GridSearchTreeRegressor_vf.pkl','wb') as f:
    pkl.dump(clf,f)

In [ ]:
#Saved down the best performing model based on the results of the grid search
best_rf_model = clf.best_estimator_

with open(notebook_dir /'best_rf_model.pkl', 'wb') as f:
    pickle.dump(best_rf_model, f)

In [ ]:
#Save down the features used in the model to maintain order
with open(notebook_dir /'GridSearchTreeFeatures_vf.pkl','wb') as f:
    pkl.dump(df_x.columns,f)

Start running cells again here to bring back in information for best performing model

In [ ]:
# run to load all saved files back into notebook
with open(notebook_dir /'GridSearchTreeRegressor_vf.pkl','rb') as f:
    clf = pkl.load(f)

with open(notebook_dir /'best_rf_model.pkl','rb') as f:
    best_rf_model = pkl.load(f)    

with open(notebook_dir /'GridSearchTreeFeatures_vf.pkl','rb') as f:
    cols = pkl.load(f)
    

In [ ]:
#Look at best parameters determined by the grid search and the score associated with that model
print(clf.best_params_)
print(clf.best_score_)

In [ ]:
#Look at the most important features in the best performing random forest model
feature_importance = pd.Series(best_rf_model.feature_importances_,index=cols)
feature_importance.sort_values(ascending=False)

### SHAP

In [ ]:
import shap

#Generate SHAP values
explainer = shap.TreeExplainer(best_rf_model)
shap_values = explainer.shap_values(x_test)

In [ ]:
#Create plot and save using Matplotlib
shap.summary_plot(shap_values, x_test, show=False)
plt.savefig('shap_summary_plot.png', bbox_inches='tight', dpi=300)

Create predicted dataset and values

In [ ]:
df_pred = df_raw.copy()

PRED_COL = "pred_est_ttl_comp_cost"

pred_log = best_rf_model.predict(df_x) 
df_pred[PRED_COL] = np.expm1(pred_log)  # back to dollars
base = df_pred[PRED_COL].astype(float).values
n = len(df_pred)


In [11]:
df_pred.head()

,ID,Crash ID,crash_fatal_fl,case_id,rpt_block_num,rpt_street_name,rpt_street_sfx,crash_speed_limit,road_constr_zone_fl,latitude,...,e-scooter_involved,other_involved,passenger car_involved,bicycle_involved,large passenger vehicle_involved,train_involved,motor vehicle_involved,motorcycle_involved,pedestrian_involved,pred_est_ttl_comp_cost
0,246606,16209332.0,False,180010160,12300,HARRIS BRANCH,PKWY,45.0,False,30.368405,...,False,False,True,False,False,False,False,False,False,74499.059298
1,245426,16182659.0,False,180010462,8300,BEN WHITE,BLVD,55.0,False,30.222877,...,False,False,True,False,False,False,False,False,False,68720.828758
2,245513,16185981.0,False,180010550,5800,LARK CREEK DR,DR,30.0,False,30.192214,...,False,False,True,False,False,False,False,False,False,31753.406108
3,245368,16180819.0,False,180010555,1700,E PARMER LN,LN,60.0,False,30.386948,...,False,False,True,False,False,False,False,False,False,67670.400072
4,278237,17979600.0,False,180010873,700,PARMER,LN,60.0,False,30.405337,...,False,False,True,False,False,False,False,False,False,114588.331731


Booleans (already True/False) with safe defaults if column missing

In [30]:
def bcol(name, default=False):
    return df_pred.get(name, pd.Series(default, index=df_pred.index)).fillna(default).astype(bool).values

onsys   = bcol("onsys_fl", True)
private = bcol("private_dr_fl", False)
workzone= bcol("road_constr_zone_fl", False)
fatal   = bcol("crash_fatal_fl", False)


loc_group = df_pred.get("Location group", pd.Series(1, index=df_pred.index)).fillna(1)
loc_group = pd.to_numeric(loc_group, errors="coerce").fillna(1).astype(int).values
freeway = (loc_group == 2)

ped     = bcol("pedestrian_involved", False)
bike    = bcol("bicycle_involved", False)
micro   = bcol("micromobility device_involved", False)
scooter = bcol("e-scooter_involved", False)
moto    = bcol("motorcycle_involved", False)

# --- Numeric fields with safe defaults ---
speed = df_pred.get("crash_speed_limit", pd.Series(30, index=df_pred.index)).astype(float).fillna(30).values
hour  = df_pred.get("hour_of_day", pd.Series(12, index=df_pred.index)).astype(int).fillna(12).values
night = (hour <= 5) | (hour >= 20)

vru = ped | bike | micro | scooter  # vulnerable road users proxy

Action Catalog

In [31]:
ACTION_DEFS = {
    "no_change": {
        "impl_cost": 0,
        "penalty": 0,
        "feasible": np.ones(n, dtype=bool),
        "mult_fn": lambda: np.ones(n, dtype=float) * 1.00,
        "why": "Predicted risk is low or interventions are not cost-effective/feasible here."
    },

    "increase_enforcement": {
        "impl_cost": 15000,
        "penalty": 4000,
        "feasible": onsys & (~private) & (~workzone),
        "mult_fn": lambda: np.clip(
            0.80
            - 0.05 * night.astype(float)
            - 0.04 * np.clip((speed - 25) / 20, 0, 1),
            0.60, 1.05
        ),
        "why": "Compliance-focused action; strongest in higher-speed or nighttime contexts."
    },

    "reduce_speed_limit": {
        "impl_cost": 8000,
        "penalty": 2000,
        "feasible": onsys & (~private) & (speed >= 25),
        "mult_fn": lambda: np.clip(
            0.90
            - 0.08 * np.clip((speed - 30) / 20, 0, 1)
            - 0.03 * vru.astype(float),
            0.65, 1.05
        ),
        "why": "Reduces severity, especially in higher-speed environments and where VRUs are present."
    },

    "add_speed_bumps": {
        "impl_cost": 60000,
        "penalty": 12000,  # emergency response/noise tradeoff proxy
        "feasible": onsys & (~private) & (~workzone) & (speed <= 35),
        "mult_fn": lambda: np.clip(
            0.78
            - 0.06 * vru.astype(float)
            + 0.06 * np.clip((speed - 30) / 10, 0, 1)   # less ideal as speed approaches 35
            + 0.04 * moto.astype(float),                # not ideal for motorcycles
            0.60, 1.10
        ),
        "why": "High-impact traffic calming where roadway context supports vertical deflection."
    },

    "improve_crosswalks": {
        "impl_cost": 35000,
        "penalty": 3000,
        # UPDATED: not feasible on freeways
        "feasible": onsys & (~private) & ped & (~freeway),
        "mult_fn": lambda: np.clip(
            0.82
            - 0.10 * ped.astype(float)
            - 0.04 * fatal.astype(float)
            - 0.04 * np.clip((speed - 30) / 20, 0, 1),
            0.55, 1.05
        ),
        "why": "Pedestrian involvement suggests reducing conflict points and improving visibility/control (not applied on freeways)."
    },

    "protected_bike_infra": {
        "impl_cost": 80000,
        "penalty": 5000,
        "feasible": onsys & (~private) & bike,
        "mult_fn": lambda: np.clip(
            0.84
            - 0.10 * bike.astype(float)
            - 0.03 * fatal.astype(float)
            - 0.02 * np.clip((speed - 25) / 25, 0, 1),
            0.58, 1.05
        ),
        "why": "Bicycle involvement supports separated/protected infrastructure to reduce conflicts."
    },

    "micromobility_zone_controls": {
        "impl_cost": 20000,
        "penalty": 2500,
        "feasible": onsys & (~private) & (micro | scooter),
        "mult_fn": lambda: np.clip(
            0.88
            - 0.10 * (micro | scooter).astype(float)
            - 0.03 * night.astype(float),
            0.60, 1.05
        ),
        "why": "Micromobility involvement supports targeted controls, education, and geo-fenced speed policies."
    },

    "work_zone_controls": {
        "impl_cost": 25000,
        "penalty": 1500,
        "feasible": workzone,
        "mult_fn": lambda: np.clip(
            0.86
            - 0.12 * workzone.astype(float)
            - 0.04 * fatal.astype(float),
            0.55, 1.05
        ),
        "why": "Work zone flag indicates temporary controls (signage, barriers, speed management) are most appropriate."
    },

    "improve_lighting": {
        "impl_cost": 30000,
        "penalty": 1000,
        "feasible": onsys & (~private) & night,
        "mult_fn": lambda: np.clip(
            0.92
            - 0.10 * night.astype(float)
            - 0.04 * vru.astype(float),
            0.60, 1.05
        ),
        "why": "Nighttime conditions suggest visibility improvements to reduce conflicts and late detection."
    },

    "signal_timing_or_signage": {
        "impl_cost": 12000,
        "penalty": 800,
        "feasible": onsys & (~private),
        "mult_fn": lambda: np.clip(
            0.94
            - 0.04 * np.clip((speed - 30) / 20, 0, 1)
            - 0.02 * vru.astype(float),
            0.70, 1.05
        ),
        "why": "Lower-cost systemic improvements appropriate when no single VRU/work-zone trigger dominates."
    }
}

action_names = list(ACTION_DEFS.keys())

Assign best actions, expected results, and justification to each row

In [32]:
# Build matrices (n_rows, n_actions)
mults = np.column_stack([ACTION_DEFS[a]["mult_fn"]() for a in action_names])
feas  = np.column_stack([ACTION_DEFS[a]["feasible"] for a in action_names])

impl_costs = np.array([ACTION_DEFS[a]["impl_cost"] for a in action_names], dtype=float)
penalties  = np.array([ACTION_DEFS[a]["penalty"] for a in action_names], dtype=float)

# Expected crash cost after action
expected = base[:, None] * mults

# Infeasible -> cannot be selected
expected = np.where(feas, expected, np.inf)

# Total score = crash cost + implementation + penalty
total = expected + impl_costs[None, :] + penalties[None, :]

best_idx = np.argmin(total, axis=1)
df_pred["best_action"] = np.array(action_names, dtype=object)[best_idx]
df_pred["expected_cost_after_action"] = expected[np.arange(n), best_idx]
df_pred["expected_total_cost_after_action"] = total[np.arange(n), best_idx]

df_pred["expected_reduction_amount"] = df_pred[PRED_COL] - df_pred["expected_cost_after_action"]
df_pred["pct_reduction"] = np.where(df_pred[PRED_COL] > 0, df_pred["expected_reduction_amount"] / df_pred[PRED_COL], 0.0)

# Rationale: action-specific + triggered context
base_why = {a: ACTION_DEFS[a]["why"] for a in action_names}

def row_rationale(r):
    a = r["best_action"]
    notes = [base_why.get(a, "Selected based on lowest total expected score given constraints and tradeoffs.")]
    if r.get("road_constr_zone_fl", False): notes.append("Work zone context detected.")
    if r.get("pedestrian_involved", False): notes.append("Pedestrian involvement detected.")
    if r.get("bicycle_involved", False): notes.append("Bicycle involvement detected.")
    if r.get("micromobility device_involved", False) or r.get("e-scooter_involved", False): notes.append("Micromobility involvement detected.")
    if r.get("crash_fatal_fl", False): notes.append("Fatality flag increases priority for higher-impact interventions.")
    if r.get("hour_of_day", 12) <= 5 or r.get("hour_of_day", 12) >= 20: notes.append("Nighttime conditions detected.")
    if r.get("crash_speed_limit", 30) >= 40: notes.append("Higher-speed environment detected.")
    return " ".join(notes)

df_pred["ai_rationale"] = df_pred.apply(row_rationale, axis=1)

In [33]:

cols_for_bi = ["latitude", "longitude", "Address", "act_est_ttl_comp_cost",
               "pred_est_ttl_comp_cost", "best_action", 
               "expected_cost_after_action", "expected_reduction_amount", 
               "pct_reduction", "ai_rationale"]

# Output BI-ready subset (edit as needed)
# cols_for_bi = [
#     "ID", "Crash ID", "latitude", "longitude",
#     "Estimated Total Comprehensive Cost", PRED_COL,
#     "crash_speed_limit", "road_constr_zone_fl", "onsys_fl", "private_dr_fl",
#     "day_of_week", "hour_of_day",
#     "pedestrian_involved", "bicycle_involved", "micromobility device_involved", "e-scooter_involved",
#     "best_action", "alt_action", "decision_margin_pct",
#     "expected_cost_after_action", "expected_total_cost_after_action",
#     "expected_reduction_amount", "pct_reduction", "ai_rationale"
# ]

df_prescriptive = df_pred[[c for c in cols_for_bi if c in df_pred.columns]].copy()

df_prescriptive.head()


,latitude,longitude,Address,act_est_ttl_comp_cost,pred_est_ttl_comp_cost,best_action,expected_cost_after_action,expected_reduction_amount,pct_reduction,ai_rationale
0,30.368405,-97.613322,N HARRIS BRANCH PKWY & E PARMER LN,60000.0,74499.059298,increase_enforcement,52894.332101,21604.727196,0.29,Compliance-focused action; strongest in higher...
1,30.222877,-97.679100,8300 E STATE HIGHWAY 71,20000.0,68720.828758,reduce_speed_limit,56351.079581,12369.749176,0.18,"Reduces severity, especially in higher-speed e..."
2,30.192214,-97.731125,5800 LARK CREEK DR,20000.0,31753.406108,no_change,31753.406108,0.000000,0.00,Predicted risk is low or interventions are not...
3,30.386948,-97.648715,E PARMER LN & DESSAU RD,60000.0,67670.400072,reduce_speed_limit,55489.728059,12180.672013,0.18,"Reduces severity, especially in higher-speed e..."
4,30.405337,-97.664316,700 E FM 734,40000.0,114588.331731,reduce_speed_limit,93962.432019,20625.899712,0.18,"Reduces severity, especially in higher-speed e..."


In [ ]:


ts = datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = output_dir / f"df_prescriptive_final_{ts}.csv"

df_prescriptive.to_csv(output_path, index=False)
print(f"Saved prescriptive output to: {output_path}")


Saved prescriptive output to: C:\Users\jacqueline.pielli\Documents\Project Nova\BI to AI\df_prescriptive_final_20260204_102224.csv


: 